# EXP-002 — CodeBERT fine-tuning trên Kaggle GPU

**Setup 1 lần trên Kaggle:** Settings → Accelerator = **GPU**, Internet = **ON**.
Cell 1: điền GH_TOKEN + HF_TOKEN (Secrets hoặc dán tay). Cell 2: sửa REPO rồi chạy từng cell.

In [ ]:
import os

# Cách 1 (khuyên dùng): Kaggle Add-ons → Secrets → thêm GITHUB_TOKEN, HF_TOKEN, attach vào notebook
# Cách 2: dán trực tiếp dưới đây (đừng share notebook này)
GITHUB_TOKEN = ""  # GitHub PAT — bắt buộc nếu repo PRIVATE
HF_TOKEN = ""  # HuggingFace token — tải dataset ổn định + tăng rate limit
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    GITHUB_TOKEN = GITHUB_TOKEN or _s.get_secret("GITHUB_TOKEN")
    HF_TOKEN = HF_TOKEN or _s.get_secret("HF_TOKEN")
except Exception as e:
    print("dùng token dán tay:", e)
os.environ["HF_TOKEN"] = HF_TOKEN  # huggingface_hub/datasets tự dùng biến này
print("GITHUB_TOKEN:", "OK" if GITHUB_TOKEN else "TRỐNG", "| HF_TOKEN:", "OK" if HF_TOKEN else "TRỐNG")


In [ ]:
REPO = "YOUR_USER/YOUR_REPO"  # <-- SỬA DÒNG NÀY
url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git" if GITHUB_TOKEN else f"https://github.com/{REPO}.git"
!git clone $url project
!git -C project remote set-url origin https://github.com/$REPO.git  # xóa token khỏi remote
%cd project
!git log --oneline -3

In [ ]:
# Kaggle đã có sẵn torch CUDA — chỉ cài phần còn thiếu, không đụng torch
!pip install -q transformers datasets scikit-learn pyyaml tqdm tree-sitter
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

In [ ]:
# Rebuild dataset foundation POJ-104 trên Kaggle (nhanh, ~1 phút) + verify leakage
!python scripts/build_dataset.py --config configs/pairs/poj104.yaml

In [ ]:
# Preflight: kiểm tra GPU/VRAM/model/config trước train
!python -m src.experiments.exp002_codebert --config configs/exp002_codebert.yaml --mode preflight --device cuda

In [ ]:
# Smoke: 1 epoch subset nhỏ — PASS mới train full
!python -m src.experiments.exp002_codebert --config configs/exp002_codebert.yaml --mode smoke --device cuda

In [ ]:
# FULL TRAINING (train mode tự chạy final evaluation cuối)
!python -m src.experiments.exp002_codebert --config configs/exp002_codebert.yaml --mode train --device cuda

In [ ]:
# Đóng gói kết quả để tải về (trừ checkpoint nặng — tải riêng nếu cần)
!tar -czf exp002_results.tar.gz results/exp002_codebert_poj104 --exclude='*.pt'
!ls -lh exp002_results.tar.gz results/exp002_codebert_poj104/
!cat results/exp002_codebert_poj104/metrics.json